# Shrub Detection — SAM Boundary Delineation + Structural Features

| | |
|---|---|
| **Author** | SAMI BAHIG |
| **Challenge** | Shrubwise Data Challenge |
| **Notebook** | `05_sam_segmentation.ipynb` |
| **Pipeline** | Setup → SAM Load → Auto Masks → GT Matching → Structural Features → Classification → Comparison |

---

This notebook introduces **Segment Anything Model (SAM)** as a zero-shot boundary delineator for shrub mapping.
Rather than relying on spectral signal alone, we exploit **structural/morphological features** extracted from
SAM-proposed object boundaries as the primary discriminative signal.

**Hypothesis:** Shrubs occupy a distinctive morphological niche — compact, roughly convex, mid-scale objects —
that SAM can detect without any shrub-specific training, and whose structural signature separates them from
soil, grass, rock, and tree canopy more reliably than NDVI alone.

**Pipeline overview:**
```
NAIP RGB (0.6 m/px)  ─── SAM Automatic Mask Generator ───► candidate object masks
                                                                   │
                                             IoU-match vs TLS ground truth masks
                                                                   │
                                    Extract structural features per mask:
                                    area · perimeter · compactness · elongation
                                    convexity · aspect_ratio · solidity
                                    fractal_dim · hu_moments (7) · SAM iou_pred
                                                                   │
                                        XGBoost / Random Forest classifier
                                                                   │
                                           Shrub mask reconstruction + metrics
```

| Approach | Primary Signal | Expected IoU |
|---|---|---|
| NDVI threshold (baseline) | spectral | ~0.19 |
| RF/XGB on 12 spectral channels | spectral | ~0.65–0.70 |
| **SAM + structural features (this notebook)** | **morphological** | **TBD** |
| ResNet34-UNet 192×192 (best) | spectral + texture | 0.8397 |

---

**References:**
- Kirillov et al. (2023). *Segment Anything.* ICCV 2023. https://arxiv.org/abs/2304.02643
- SAM repository: https://github.com/facebookresearch/segment-anything
- ShrubMap pipeline: https://github.com/samibahig/ShrubMap-Data-Challenge

In [ ]:
# ── CELL 1 : Environment Setup ────────────────────────────────────────────────
# Installs SAM and all required dependencies.
# SAM checkpoint (~375 MB) is downloaded once and cached.
# Runtime: ~3 min on first run, ~30 s on subsequent runs (cached checkpoint).

import subprocess, sys, os
from pathlib import Path

# ── Core packages ─────────────────────────────────────────────────────────────
_pkgs = [
    'torch', 'torchvision',
    'rasterio', 'shapely', 'pyproj', 'geopandas',
    'scikit-image', 'scikit-learn',
    'xgboost', 'imbalanced-learn',
    'matplotlib', 'seaborn', 'pandas', 'numpy', 'tqdm', 'shap',
]
for pkg in _pkgs:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', pkg], check=False)

# ── Segment Anything Model ────────────────────────────────────────────────────
# Install from GitHub source (no PyPI release as of 2024).
subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q',
     'git+https://github.com/facebookresearch/segment-anything.git'],
    check=False
)

# ── Download SAM checkpoint (ViT-H, default — highest accuracy) ───────────────
# Alternatives: vit_l (~1.2 GB), vit_b (~375 MB — fastest, use on CPU)
CKPT_DIR  = Path('/home/jovyan/work/_User-Persistent-Storage_CephBlock_/sam_checkpoints')
CKPT_DIR.mkdir(parents=True, exist_ok=True)

# We default to vit_b for CPU environments; swap URL for vit_h on GPU.
import torch
ON_GPU = torch.cuda.is_available()

if ON_GPU:
    CKPT_URL  = 'https://dl.fbaipublicfiles.com/segment_anything/sam_vit_h_4b8939.pth'
    CKPT_FILE = CKPT_DIR / 'sam_vit_h_4b8939.pth'
    SAM_MODEL_TYPE = 'vit_h'
else:
    CKPT_URL  = 'https://dl.fbaipublicfiles.com/segment_anything/sam_vit_b_01ec64.pth'
    CKPT_FILE = CKPT_DIR / 'sam_vit_b_01ec64.pth'
    SAM_MODEL_TYPE = 'vit_b'

if not CKPT_FILE.exists():
    print(f'Downloading SAM checkpoint ({SAM_MODEL_TYPE}) — this may take a few minutes …')
    subprocess.run(['wget', '-q', '-O', str(CKPT_FILE), CKPT_URL], check=True)
    print('Download complete ✓')
else:
    print(f'SAM checkpoint cached at {CKPT_FILE} ✓')

print(f'\nEnvironment ready ✓')
print(f'  GPU available : {ON_GPU}')
print(f'  SAM model     : {SAM_MODEL_TYPE}')

In [ ]:
# ── CELL 2 : Imports ──────────────────────────────────────────────────────────

import os, json, time, random, warnings
from pathlib import Path
from typing import Dict, List, Tuple, Optional

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns

import torch
import rasterio
from rasterio.transform import from_bounds
from rasterio.enums import Resampling

from skimage import measure, morphology
from skimage.measure import regionprops, regionprops_table
from skimage.morphology import convex_hull_image
from scipy.ndimage import label as nd_label
from scipy.spatial import ConvexHull

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.metrics import (
    jaccard_score, f1_score, recall_score, precision_score,
    confusion_matrix, classification_report,
)
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
import xgboost as xgb
import shap
from tqdm import tqdm

# SAM
from segment_anything import sam_model_registry, SamAutomaticMaskGenerator, SamPredictor

warnings.filterwarnings('ignore')
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print('Imports OK ✓')
print(f'  PyTorch  : {torch.__version__}')
print(f'  Device   : {DEVICE}')
print(f'  XGBoost  : {xgb.__version__}')
print(f'  SAM      : segment_anything imported ✓')

In [ ]:
# ── CELL 3 : Configuration ────────────────────────────────────────────────────
# All paths, site names, and hyperparameters are centralised here.
# Modify this cell only to adapt to a different storage layout.

# ── Storage root ──────────────────────────────────────────────────────────────
WORK = Path('/home/jovyan/work/_User-Persistent-Storage_CephBlock_')
# NOTE: For Colab / Docker, replace with a relative path:
# WORK = Path('.')

# ── Input data ────────────────────────────────────────────────────────────────
NAIP_DIR    = WORK / 'sprint4' / 'naip'          # NAIP GeoTIFFs  (4-band, 0.6 m/px)
MASK_DIR    = WORK / 'sprint4' / 'masks'         # TLS binary masks (GeoTIFF, 0/1)
PATCH_ROOT  = WORK / 'sprint4' / 'patches'       # 64×64 patches from notebook 01

# ── SAM checkpoint ────────────────────────────────────────────────────────────
CKPT_DIR       = WORK / 'sam_checkpoints'
SAM_MODEL_TYPE = 'vit_b'   # vit_b (CPU-friendly) | vit_l | vit_h (GPU recommended)
_ckpt_map = {
    'vit_h': 'sam_vit_h_4b8939.pth',
    'vit_l': 'sam_vit_l_0b3195.pth',
    'vit_b': 'sam_vit_b_01ec64.pth',
}
CKPT_FILE = CKPT_DIR / _ckpt_map[SAM_MODEL_TYPE]

# ── Output directory ──────────────────────────────────────────────────────────
OUT_DIR = WORK / 'sprint4' / 'sam_features'
OUT_DIR.mkdir(parents=True, exist_ok=True)

# ── Six California field sites ────────────────────────────────────────────────
SITES = [
    'sedgwick',
    'cr',           # Coal Oil Point Reserve
    'jasper',       # Jasper Ridge Biological Preserve
    'hopland',      # Hopland Research & Extension Center
    'dl-bliss',     # DL Bliss State Park
    'carrizo',      # Carrizo Plain National Monument
]

# Maps site slug → NAIP filename (mirrors 01_data_preparation convention)
SITE_NAIP = {
    'sedgwick': 'sedgwick.tif',
    'cr':       'cr.tif',
    'jasper':   'jasper.tif',
    'hopland':  'hopland.tif',
    'dl-bliss': 'dl-bliss.tif',
    'carrizo':  'carrizo.tif',
}
SITE_MASK = {s: f'{s}_mask.tif' for s in SITES}

# ── SAM automatic mask generator hyperparameters ──────────────────────────────
# Tuned for 0.6 m/px NAIP imagery and typical shrub sizes (0.5–5 m diameter).
SAM_AMG_KWARGS = dict(
    points_per_side        = 32,    # Grid density — increase to 64 on GPU
    pred_iou_thresh        = 0.86,  # Keep only high-confidence masks
    stability_score_thresh = 0.92,  # Remove unstable boundary predictions
    crop_n_layers          = 1,     # One crop layer captures sub-patch objects
    crop_n_points_downscale_factor = 2,
    min_mask_region_area   = 50,    # Drop masks < 50 px² (~18 cm² at 0.6 m/px)
)

# ── Structural feature extraction settings ────────────────────────────────────
MIN_MASK_PX   = 50     # Minimum mask area in pixels (< 50 px = noise)
MAX_MASK_PX   = 50000  # Maximum mask area (> 50000 px = likely background)
IOU_MATCH_THR = 0.30   # IoU threshold to label a SAM mask as "shrub" vs TLS GT

# ── Classifier settings ───────────────────────────────────────────────────────
CV_FOLDS = 5

print('Configuration loaded ✓')
print(f'  SAM model      : {SAM_MODEL_TYPE}')
print(f'  Checkpoint     : {CKPT_FILE}')
print(f'  Checkpoint OK  : {CKPT_FILE.exists()}')
print(f'  Sites          : {SITES}')
print(f'  Output dir     : {OUT_DIR}')

In [ ]:
# ── CELL 4 : Load SAM Model ───────────────────────────────────────────────────
# Loads the SAM checkpoint and instantiates the Automatic Mask Generator.
# The AMG runs SAM in a dense grid-of-points mode — no manual prompts needed.

print(f'Loading SAM ({SAM_MODEL_TYPE}) …')
t0 = time.time()

sam = sam_model_registry[SAM_MODEL_TYPE](checkpoint=str(CKPT_FILE))
sam.to(device=DEVICE)
sam.eval()

mask_generator = SamAutomaticMaskGenerator(sam, **SAM_AMG_KWARGS)

print(f'SAM loaded in {time.time()-t0:.1f} s ✓')
print(f'  Model type : {SAM_MODEL_TYPE}')
print(f'  Device     : {DEVICE}')
print(f'  AMG config : points_per_side={SAM_AMG_KWARGS["points_per_side"]}, '
      f'pred_iou_thresh={SAM_AMG_KWARGS["pred_iou_thresh"]}')

In [ ]:
# ── CELL 5 : Helper Functions ─────────────────────────────────────────────────
# Stateless utilities for NAIP loading, mask I/O, and feature extraction.
# All functions are independent and can be imported in other notebooks.

def load_naip_rgb(naip_path: Path) -> Tuple[np.ndarray, object]:
    """Load NAIP GeoTIFF and return (H, W, 3) uint8 RGB array + rasterio profile.

    SAM expects uint8 RGB. NAIP uint16 values are linearly scaled to [0, 255]
    using the 2nd–98th percentile of each band to preserve contrast.

    Args:
        naip_path: Path to the 4-band NAIP GeoTIFF.

    Returns:
        rgb      : (H, W, 3) uint8 array — bands [R, G, B].
        profile  : rasterio dataset profile for georeferenced output.
    """
    with rasterio.open(naip_path) as src:
        # Bands: 1=R, 2=G, 3=B, 4=NIR (NAIP convention)
        data    = src.read([1, 2, 3]).astype(np.float32)   # (3, H, W)
        profile = src.profile

    rgb = np.zeros_like(data, dtype=np.float32)
    for i in range(3):
        p2, p98 = np.percentile(data[i], [2, 98])
        rgb[i]  = np.clip((data[i] - p2) / (p98 - p2 + 1e-6), 0, 1)

    rgb_uint8 = (rgb * 255).astype(np.uint8).transpose(1, 2, 0)  # (H, W, 3)
    return rgb_uint8, profile


def load_tls_mask(mask_path: Path, target_shape: Tuple[int, int]) -> np.ndarray:
    """Load TLS binary shrub mask and resize to match NAIP spatial extent.

    Args:
        mask_path    : Path to the TLS-derived binary mask GeoTIFF.
        target_shape : (H, W) to match the NAIP image.

    Returns:
        mask : (H, W) uint8 binary array (0 = background, 1 = shrub).
    """
    H, W = target_shape
    with rasterio.open(mask_path) as src:
        mask = src.read(
            1,
            out_shape=(H, W),
            resampling=Resampling.nearest,
        ).astype(np.uint8)
    return (mask > 0).astype(np.uint8)


def masks_iou(mask_a: np.ndarray, mask_b: np.ndarray) -> float:
    """Compute binary IoU between two 2-D boolean arrays."""
    intersection = np.logical_and(mask_a, mask_b).sum()
    union        = np.logical_or(mask_a, mask_b).sum()
    return float(intersection) / float(union + 1e-9)


def extract_structural_features(binary_mask: np.ndarray, sam_meta: dict) -> dict:
    """Extract morphological/structural features from a single binary mask.

    Features are computed on the connected component corresponding to the
    largest region in the mask to handle occasional multi-blob SAM outputs.

    Args:
        binary_mask : (H, W) boolean or 0/1 array for a single object mask.
        sam_meta    : SAM output dict for this mask (contains iou_pred,
                      stability_score, area, bbox, point_coords).

    Returns:
        feats : Dict of scalar feature values. Returns NaN-filled dict if
                the mask is empty or smaller than MIN_MASK_PX.
    """
    m = binary_mask.astype(bool)
    total_px = m.sum()

    _nan = {k: np.nan for k in [
        'area_px', 'perimeter', 'compactness', 'elongation',
        'aspect_ratio', 'solidity', 'convexity', 'extent',
        'fractal_dim', 'eccentricity', 'equiv_diameter',
        'hu0', 'hu1', 'hu2', 'hu3', 'hu4', 'hu5', 'hu6',
        'sam_iou_pred', 'sam_stability_score', 'sam_area',
    ]}

    if total_px < MIN_MASK_PX:
        return _nan

    # ── Largest connected component ───────────────────────────────────────────
    labeled, n_cc = nd_label(m)
    if n_cc == 0:
        return _nan
    cc_sizes = [(labeled == i).sum() for i in range(1, n_cc + 1)]
    largest  = (labeled == (np.argmax(cc_sizes) + 1))

    # ── Region properties via scikit-image ───────────────────────────────────
    props = regionprops(largest.astype(np.uint8))[0]

    area       = props.area                              # pixels
    perimeter  = props.perimeter                         # pixels (Crofton formula)
    ecc        = props.eccentricity                      # 0 = circle, 1 = line
    eq_diam    = props.equivalent_diameter               # diameter of equal-area circle
    extent     = props.extent                            # area / bounding_box_area
    solidity   = props.solidity                          # area / convex_hull_area
    maj_ax     = props.major_axis_length
    min_ax     = props.minor_axis_length + 1e-9

    # ── Derived shape descriptors ─────────────────────────────────────────────
    compactness = (4 * np.pi * area) / (perimeter ** 2 + 1e-9)  # 1.0 = perfect circle
    elongation  = maj_ax / (min_ax)                              # 1.0 = square
    aspect_ratio = maj_ax / (min_ax)                             # same as elongation

    # Convexity: perimeter(convex_hull) / perimeter(mask)
    try:
        chull     = convex_hull_image(largest)
        ch_props  = regionprops(chull.astype(np.uint8))[0]
        convexity = ch_props.perimeter / (perimeter + 1e-9)
    except Exception:
        convexity = np.nan

    # Fractal dimension (box-counting approximation)
    # D ≈ log(N(ε)) / log(1/ε) slope over binary boundary image
    try:
        boundary = measure.find_contours(largest.astype(float), 0.5)
        if boundary:
            pts = np.concatenate(boundary, axis=0)
            counts, box_sizes = [], []
            for eps in [2, 4, 8, 16]:
                grid_pts = (pts / eps).astype(int)
                counts.append(len(set(map(tuple, grid_pts))))
                box_sizes.append(1.0 / eps)
            log_eps    = np.log(np.array(box_sizes))
            log_counts = np.log(np.array(counts) + 1)
            if log_eps.std() > 0:
                fractal_dim = np.polyfit(log_eps, log_counts, 1)[0]
            else:
                fractal_dim = np.nan
        else:
            fractal_dim = np.nan
    except Exception:
        fractal_dim = np.nan

    # Hu moments (log-scale, rotation/scale invariant shape descriptors)
    hu = props.moments_hu                                # array of 7
    hu_log = np.sign(hu) * np.log10(np.abs(hu) + 1e-30)

    return {
        # ── Size ────────────────────────────────────────────────────────────
        'area_px'          : float(area),
        'equiv_diameter'   : float(eq_diam),
        # ── Boundary complexity ─────────────────────────────────────────────
        'perimeter'        : float(perimeter),
        'compactness'      : float(compactness),
        'convexity'        : float(convexity),
        'fractal_dim'      : float(fractal_dim) if not np.isnan(fractal_dim) else np.nan,
        # ── Shape / orientation ─────────────────────────────────────────────
        'elongation'       : float(elongation),
        'aspect_ratio'     : float(aspect_ratio),
        'eccentricity'     : float(ecc),
        'solidity'         : float(solidity),
        'extent'           : float(extent),
        # ── Hu moments (7) ──────────────────────────────────────────────────
        'hu0': float(hu_log[0]), 'hu1': float(hu_log[1]),
        'hu2': float(hu_log[2]), 'hu3': float(hu_log[3]),
        'hu4': float(hu_log[4]), 'hu5': float(hu_log[5]),
        'hu6': float(hu_log[6]),
        # ── SAM confidence signals ──────────────────────────────────────────
        'sam_iou_pred'        : float(sam_meta.get('predicted_iou', np.nan)),
        'sam_stability_score' : float(sam_meta.get('stability_score', np.nan)),
        'sam_area'            : float(sam_meta.get('area', np.nan)),
    }


print('Helper functions defined ✓')
print('  load_naip_rgb, load_tls_mask, masks_iou, extract_structural_features')

In [ ]:
# ── CELL 6 : SAM Inference — Generate Candidate Masks ─────────────────────────
# Runs the SAM Automatic Mask Generator on the RGB NAIP tile for each site.
# SAM proposes object segments at multiple scales without any domain prompts.
#
# Runtime estimate:
#   GPU (A100)  : ~20 s/site
#   CPU (16 GB) : ~4–8 min/site  (reduce points_per_side to 16 if too slow)
#
# Output per site:
#   sam_results[site] = list of dicts, each with keys:
#     segmentation      : (H, W) bool array
#     area              : int (pixels)
#     bbox              : [x, y, w, h]
#     predicted_iou     : float (SAM's self-estimated mask quality)
#     stability_score   : float (mask robustness to threshold shift)
#     point_coords      : [[x, y]] (prompt point used to generate this mask)

sam_results : Dict[str, list] = {}
site_rgb    : Dict[str, np.ndarray] = {}
site_gt     : Dict[str, np.ndarray] = {}

for site in SITES:
    naip_path = NAIP_DIR / SITE_NAIP[site]
    mask_path = MASK_DIR / SITE_MASK[site]

    if not naip_path.exists():
        print(f'[{site}] NAIP file not found — skipping ({naip_path})')
        continue

    print(f'[{site}] Loading NAIP …', end=' ', flush=True)
    rgb, profile = load_naip_rgb(naip_path)
    H, W = rgb.shape[:2]
    site_rgb[site] = rgb
    print(f'{H}×{W} px loaded', end=' | ', flush=True)

    if mask_path.exists():
        gt = load_tls_mask(mask_path, (H, W))
    else:
        gt = np.zeros((H, W), dtype=np.uint8)
        print(f'[WARN] GT mask not found — labels will be empty', end=' | ')
    site_gt[site] = gt

    print('Running SAM …', end=' ', flush=True)
    t0 = time.time()
    with torch.inference_mode():
        masks = mask_generator.generate(rgb)
    elapsed = time.time() - t0

    # Filter by size bounds
    masks = [m for m in masks
             if MIN_MASK_PX <= m['area'] <= MAX_MASK_PX]

    sam_results[site] = masks
    print(f'{len(masks)} masks in {elapsed:.0f}s ✓')

print('\nSAM inference complete ✓')
print(f'  Total candidate masks : {sum(len(v) for v in sam_results.values()):,}')

In [ ]:
# ── CELL 7 : Ground-Truth Matching ────────────────────────────────────────────
# Labels each SAM candidate mask as shrub (1) or background (0) by comparing
# it against the TLS-derived ground-truth mask with an IoU threshold.
#
# Strategy:
#   For each SAM mask M:
#     iou = |M ∩ GT| / |M ∪ GT|
#     label = 1 if iou >= IOU_MATCH_THR else 0
#
# This creates a weakly-supervised labelling — a SAM mask is called "shrub" if
# it overlaps substantially with at least one annotated shrub region.
#
# Note: this approach under-counts shrub masks that SAM splits across two
# proposals, and over-counts masks that cover partially-annotated canopy
# edges. Both are bounded errors at IOU_MATCH_THR=0.30.

records = []  # Will become the structural feature DataFrame

for site, masks in tqdm(sam_results.items(), desc='GT matching'):
    gt   = site_gt[site].astype(bool)
    H, W = gt.shape

    shrub_count = 0
    for i, m in enumerate(masks):
        seg = m['segmentation'].astype(bool)

        # IoU against TLS GT
        iou_gt = masks_iou(seg, gt)
        label  = int(iou_gt >= IOU_MATCH_THR)
        if label:
            shrub_count += 1

        # Extract structural features
        feats = extract_structural_features(seg, m)
        feats.update({
            'site'      : site,
            'mask_idx'  : i,
            'label'     : label,
            'iou_vs_gt' : iou_gt,
        })
        records.append(feats)

    print(f'  {site}: {len(masks)} masks → {shrub_count} shrub / {len(masks)-shrub_count} background')

df = pd.DataFrame(records)
df.to_csv(OUT_DIR / 'structural_features.csv', index=False)

print(f'\nDataset shape : {df.shape}')
print(f'Shrub masks   : {df.label.sum():,} ({100*df.label.mean():.1f}%)')
print(f'Background    : {(df.label==0).sum():,} ({100*(1-df.label.mean()):.1f}%)')
print(f'Saved to      : {OUT_DIR}/structural_features.csv')
df.head()

In [ ]:
# ── CELL 8 : Exploratory Analysis — Structural Feature Distributions ──────────
# Visualises how shrub vs background masks differ across structural dimensions.
# These plots confirm the hypothesis that shrubs occupy a distinct morphological
# niche in the SAM candidate space.

STRUCT_FEATS = [
    'area_px', 'compactness', 'elongation', 'solidity',
    'convexity', 'fractal_dim', 'eccentricity', 'equiv_diameter',
]

fig, axes = plt.subplots(2, 4, figsize=(18, 8))
axes = axes.flatten()

palette = {0: '#8C8C8C', 1: '#2E7D32'}   # gray = background, green = shrub

for ax, feat in zip(axes, STRUCT_FEATS):
    for lbl, color in palette.items():
        vals = df.loc[df.label == lbl, feat].dropna()
        if vals.empty:
            continue
        ax.hist(
            vals, bins=40, alpha=0.6, color=color,
            label='Shrub' if lbl == 1 else 'Background',
            density=True,
        )
    ax.set_title(feat.replace('_', ' ').title(), fontsize=11)
    ax.set_xlabel('Value')
    ax.set_ylabel('Density')
    ax.legend(fontsize=8)

plt.suptitle('SAM Candidate Masks — Structural Feature Distributions\n'
             'Green = Shrub (IoU ≥ 0.30 vs TLS GT) · Gray = Background',
             fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig(OUT_DIR / 'feature_distributions.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: feature_distributions.png')

In [ ]:
# ── CELL 9 : Feature Correlation Matrix ───────────────────────────────────────
# Checks for multicollinearity among structural features.
# Highly correlated pairs (|r| > 0.85) are candidates for removal
# to reduce overfitting in the downstream classifier.

FEAT_COLS = [
    'area_px', 'perimeter', 'compactness', 'elongation',
    'aspect_ratio', 'solidity', 'convexity', 'extent',
    'fractal_dim', 'eccentricity', 'equiv_diameter',
    'hu0', 'hu1', 'hu2', 'hu3',
    'sam_iou_pred', 'sam_stability_score', 'sam_area',
]

# Keep only columns that exist in the dataframe
FEAT_COLS = [c for c in FEAT_COLS if c in df.columns]

corr = df[FEAT_COLS].corr()

fig, ax = plt.subplots(figsize=(14, 12))
mask_tri = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(
    corr, mask=mask_tri, annot=True, fmt='.2f', cmap='RdBu_r',
    center=0, vmin=-1, vmax=1, linewidths=0.5, ax=ax,
    annot_kws={'size': 7},
)
ax.set_title('Structural Feature Correlation Matrix', fontsize=13)
plt.tight_layout()
plt.savefig(OUT_DIR / 'feature_correlation.png', dpi=150, bbox_inches='tight')
plt.show()

# Report highly correlated pairs
high_corr = [
    (FEAT_COLS[i], FEAT_COLS[j], corr.iloc[i, j])
    for i in range(len(FEAT_COLS))
    for j in range(i+1, len(FEAT_COLS))
    if abs(corr.iloc[i, j]) > 0.85
]
print(f'High-correlation pairs (|r| > 0.85):')
for a, b, r in sorted(high_corr, key=lambda x: -abs(x[2])):
    print(f'  {a:<25} ↔  {b:<25}  r={r:.3f}')

In [ ]:
# ── CELL 10 : Classifier — Structural Features Only ───────────────────────────
# Trains XGBoost and Random Forest on structural features ONLY.
# No spectral information is used — this tests the morphological hypothesis.
#
# Evaluation: stratified 5-fold CV on the mask-level dataset.
# Primary metric: IoU (Jaccard) on the reconstructed binary pixel mask.
#
# NOTE: The mask-level classifier produces per-mask labels (shrub / background).
# Pixel-level metrics are computed by unioning the predicted shrub masks.

# ── Prepare dataset ───────────────────────────────────────────────────────────
df_clean = df[FEAT_COLS + ['label', 'site']].dropna(subset=FEAT_COLS)
X = df_clean[FEAT_COLS].values
y = df_clean['label'].values

print(f'Dataset: {X.shape[0]:,} masks × {X.shape[1]} structural features')
print(f'Shrub   : {y.sum():,} ({100*y.mean():.1f}%)')
print(f'Backgr. : {(y==0).sum():,}')

# ── XGBoost ──────────────────────────────────────────────────────────────────
xgb_clf = xgb.XGBClassifier(
    n_estimators     = 500,
    max_depth        = 6,
    learning_rate    = 0.05,
    subsample        = 0.8,
    colsample_bytree = 0.8,
    scale_pos_weight = (y==0).sum() / (y.sum() + 1e-9),  # handle class imbalance
    use_label_encoder= False,
    eval_metric      = 'logloss',
    random_state     = SEED,
    verbosity        = 0,
)

# ── Random Forest ─────────────────────────────────────────────────────────────
rf_clf = RandomForestClassifier(
    n_estimators  = 500,
    max_depth     = None,
    class_weight  = 'balanced',
    n_jobs        = -1,
    random_state  = SEED,
)

# ── Stratified 5-fold CV ──────────────────────────────────────────────────────
cv = StratifiedKFold(n_splits=CV_FOLDS, shuffle=True, random_state=SEED)

results_cv = {}
for name, clf in [('XGBoost', xgb_clf), ('Random Forest', rf_clf)]:
    t0    = time.time()
    iou   = cross_val_score(clf, X, y, cv=cv, scoring='jaccard', n_jobs=-1)
    f1    = cross_val_score(clf, X, y, cv=cv, scoring='f1',      n_jobs=-1)
    rec   = cross_val_score(clf, X, y, cv=cv, scoring='recall',  n_jobs=-1)
    prec  = cross_val_score(clf, X, y, cv=cv, scoring='precision', n_jobs=-1)
    elapsed = time.time() - t0

    results_cv[name] = {
        'IoU (mask-level)'  : f'{iou.mean():.4f} ± {iou.std():.4f}',
        'F1'                : f'{f1.mean():.4f} ± {f1.std():.4f}',
        'Recall'            : f'{rec.mean():.4f} ± {rec.std():.4f}',
        'Precision'         : f'{prec.mean():.4f} ± {prec.std():.4f}',
        'Train time (s)'    : f'{elapsed:.0f}',
        '_iou_mean'         : iou.mean(),
    }
    print(f'\n{name}')
    print(f'  IoU   : {iou.mean():.4f} ± {iou.std():.4f}')
    print(f'  F1    : {f1.mean():.4f} ± {f1.std():.4f}')
    print(f'  Recall: {rec.mean():.4f} ± {rec.std():.4f}')
    print(f'  Time  : {elapsed:.0f}s')

In [ ]:
# ── CELL 11 : Pixel-Level Metrics — Mask Reconstruction ───────────────────────
# The classifier operates at mask level (one prediction per SAM proposal).
# To compare with the deep learning models (which produce pixel-level predictions)
# we reconstruct a pixel-level binary map by unioning all predicted-shrub masks.
#
# Leave-one-site-out evaluation mirrors the real deployment scenario where
# the model is tested on an unseen geographic site.

def pixel_metrics(y_true_px: np.ndarray, y_pred_px: np.ndarray, prefix: str = '') -> dict:
    """Compute pixel-level binary classification metrics."""
    y_true_f = y_true_px.flatten()
    y_pred_f = y_pred_px.flatten()
    return {
        f'{prefix}IoU'      : jaccard_score(y_true_f, y_pred_f, zero_division=0),
        f'{prefix}F1'       : f1_score(y_true_f, y_pred_f, zero_division=0),
        f'{prefix}Recall'   : recall_score(y_true_f, y_pred_f, zero_division=0),
        f'{prefix}Precision': precision_score(y_true_f, y_pred_f, zero_division=0),
    }


# ── Leave-one-site-out evaluation with XGBoost ────────────────────────────────
print('Leave-one-site-out pixel-level evaluation (XGBoost)\n')
loso_records = []

for test_site in sam_results.keys():
    train_sites = [s for s in sam_results.keys() if s != test_site]

    df_tr = df_clean[df_clean.site.isin(train_sites)]
    df_te = df_clean[df_clean.site == test_site]

    if df_te.empty or df_tr.empty:
        continue

    X_tr, y_tr = df_tr[FEAT_COLS].values, df_tr['label'].values
    X_te       = df_te[FEAT_COLS].values

    clf = xgb.XGBClassifier(
        n_estimators=300, max_depth=6, learning_rate=0.05,
        scale_pos_weight=(y_tr==0).sum() / (y_tr.sum() + 1e-9),
        use_label_encoder=False, eval_metric='logloss',
        random_state=SEED, verbosity=0,
    )
    clf.fit(X_tr, y_tr)
    y_pred_masks = clf.predict(X_te)

    # Reconstruct pixel-level prediction map
    gt_px   = site_gt[test_site]
    H, W    = gt_px.shape
    pred_px = np.zeros((H, W), dtype=np.uint8)

    site_masks = sam_results[test_site]
    df_te_idx  = df_clean[df_clean.site == test_site].index.tolist()
    te_rows    = df_clean.loc[df_te_idx, 'mask_idx'].values

    for pred, orig_idx in zip(y_pred_masks, te_rows):
        if pred == 1 and orig_idx < len(site_masks):
            pred_px = np.logical_or(pred_px, site_masks[orig_idx]['segmentation']).astype(np.uint8)

    m = pixel_metrics(gt_px, pred_px)
    m['site'] = test_site
    loso_records.append(m)

    print(f'  [{test_site:12s}]  IoU={m["IoU"]:.4f}  F1={m["F1"]:.4f}  '
          f'Recall={m["Recall"]:.4f}  Precision={m["Precision"]:.4f}')

loso_df = pd.DataFrame(loso_records)
if not loso_df.empty:
    print(f'\nMean across sites:')
    print(loso_df[['IoU','F1','Recall','Precision']].mean().to_string())

In [ ]:
# ── CELL 12 : SHAP Feature Importance ─────────────────────────────────────────
# Trains a final XGBoost model on the full dataset and computes SHAP values
# to identify which structural features drive shrub vs background discrimination.
#
# Key question: Is it size (area), shape compactness, or boundary complexity
# (fractal_dim, convexity) that best separates shrubs from other land cover?

# ── Final model fit ───────────────────────────────────────────────────────────
xgb_final = xgb.XGBClassifier(
    n_estimators     = 500,
    max_depth        = 6,
    learning_rate    = 0.05,
    subsample        = 0.8,
    colsample_bytree = 0.8,
    scale_pos_weight = (y==0).sum() / (y.sum() + 1e-9),
    use_label_encoder= False,
    eval_metric      = 'logloss',
    random_state     = SEED,
    verbosity        = 0,
)
xgb_final.fit(X, y)

# ── SHAP values ───────────────────────────────────────────────────────────────
explainer   = shap.TreeExplainer(xgb_final)
shap_values = explainer.shap_values(X)

fig, axes = plt.subplots(1, 2, figsize=(18, 7))

# Beeswarm summary plot
plt.sca(axes[0])
shap.summary_plot(
    shap_values, X, feature_names=FEAT_COLS,
    show=False, plot_size=None,
)
axes[0].set_title('SHAP Beeswarm — Structural Features', fontsize=12)

# Bar importance plot
mean_abs_shap = np.abs(shap_values).mean(axis=0)
importance_df = pd.DataFrame({
    'feature'    : FEAT_COLS,
    'mean_|SHAP|': mean_abs_shap,
}).sort_values('mean_|SHAP|', ascending=True)

axes[1].barh(
    importance_df['feature'], importance_df['mean_|SHAP|'],
    color='#2E7D32', edgecolor='white', height=0.7,
)
axes[1].set_xlabel('Mean |SHAP value|', fontsize=11)
axes[1].set_title('Structural Feature Importance (SHAP)', fontsize=12)
axes[1].grid(axis='x', alpha=0.3)

plt.suptitle(
    'SHAP Analysis — XGBoost on SAM Structural Features\n'
    'Which morphological properties distinguish shrubs from other objects?',
    fontsize=13, y=1.01,
)
plt.tight_layout()
plt.savefig(OUT_DIR / 'shap_structural.png', dpi=150, bbox_inches='tight')
plt.show()

print('\nTop-5 structural features by SHAP importance:')
print(importance_df.tail(5)[['feature','mean_|SHAP|']].iloc[::-1].to_string(index=False))

In [ ]:
# ── CELL 13 : Visual Inspection — SAM Boundary Overlay ───────────────────────
# Visualises SAM-delineated shrub boundaries on NAIP RGB tiles.
# For each site: (a) RGB, (b) TLS GT, (c) all SAM masks, (d) predicted-shrub masks.

def show_masks_on_image(
    ax, rgb: np.ndarray, masks: List[dict],
    pred_labels: Optional[np.ndarray] = None,
    title: str = '',
    alpha: float = 0.35,
) -> None:
    """Overlay SAM masks on an RGB image.

    Args:
        ax          : Matplotlib axis.
        rgb         : (H, W, 3) uint8 RGB array.
        masks       : List of SAM mask dicts.
        pred_labels : If provided, only masks with label==1 are shown in green;
                      masks with label==0 are shown in red.
        title       : Axis title.
        alpha       : Mask fill transparency.
    """
    ax.imshow(rgb)
    H, W = rgb.shape[:2]
    overlay = np.zeros((H, W, 4), dtype=np.float32)

    for i, m in enumerate(masks):
        seg = m['segmentation']
        if pred_labels is not None:
            color = np.array([0.18, 0.49, 0.20, alpha]) if pred_labels[i] else \
                    np.array([0.80, 0.15, 0.15, alpha * 0.4])
        else:
            color = np.array([*plt.cm.tab20(i % 20)[:3], alpha])
        overlay[seg] = color

    ax.imshow(overlay)
    if title:
        ax.set_title(title, fontsize=10)
    ax.axis('off')


# Plot one representative site (first available)
vis_sites = list(sam_results.keys())[:2]

for site in vis_sites:
    rgb   = site_rgb[site]
    gt    = site_gt[site]
    masks = sam_results[site]

    # Get per-mask predictions from the final model
    df_site = df_clean[df_clean.site == site].reset_index(drop=True)
    X_site  = df_site[FEAT_COLS].values
    pred_site = xgb_final.predict(X_site)

    # Rebuild mask-index mapping
    site_mask_idx = df_site['mask_idx'].values
    pred_labels_all = np.zeros(len(masks), dtype=int)
    for pred, midx in zip(pred_site, site_mask_idx):
        if midx < len(masks):
            pred_labels_all[midx] = pred

    fig, axes = plt.subplots(1, 4, figsize=(22, 6))

    # (a) NAIP RGB
    axes[0].imshow(rgb)
    axes[0].set_title(f'NAIP RGB — {site}', fontsize=10)
    axes[0].axis('off')

    # (b) TLS Ground Truth
    gt_rgb = np.stack([gt * 46, gt * 125, gt * 50], axis=-1).astype(np.uint8)
    axes[1].imshow(rgb)
    axes[1].imshow(
        np.concatenate([gt_rgb, (gt * 160)[..., None]], axis=-1).astype(np.uint8)
    )
    axes[1].set_title('TLS Ground Truth', fontsize=10)
    axes[1].axis('off')

    # (c) All SAM candidate masks
    show_masks_on_image(axes[2], rgb, masks, title='All SAM Candidates')

    # (d) Predicted shrub masks (green) vs background (faint red)
    show_masks_on_image(
        axes[3], rgb, masks,
        pred_labels=pred_labels_all,
        title='Predicted Shrub (green) vs Background (red)',
    )

    plt.suptitle(
        f'SAM Boundary Delineation + XGBoost Classification — {site}',
        fontsize=12, y=1.01,
    )
    plt.tight_layout()
    plt.savefig(OUT_DIR / f'sam_overlay_{site}.png', dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Saved: sam_overlay_{site}.png')

In [ ]:
# ── CELL 14 : Results Comparison Table ────────────────────────────────────────
# Collates this notebook's results against the published baselines from the
# ShrubMap pipeline. Provides a clear picture of where SAM + structural features
# sits relative to spectral-only and deep learning approaches.

# ── Published baselines from notebooks 03 and 04 ─────────────────────────────
BASELINES = [
    {'Model': 'NDVI Threshold',                'Signal': 'spectral',          'IoU': 0.19,   'F1': 0.32,   'Recall': 0.80,   'Precision': 0.19},
    {'Model': 'Random Forest (12 ch)',         'Signal': 'spectral+texture',  'IoU': 0.656,  'F1': 0.792,  'Recall': 0.82,   'Precision': 0.766},
    {'Model': 'XGBoost (12 ch)',               'Signal': 'spectral+texture',  'IoU': 0.672,  'F1': 0.804,  'Recall': 0.830,  'Precision': 0.780},
    {'Model': 'UNet 128×128',                  'Signal': 'spectral+texture',  'IoU': 0.751,  'F1': 0.858,  'Recall': 0.879,  'Precision': 0.839},
    {'Model': 'EfficientNet-B3 128×128',       'Signal': 'spectral+texture',  'IoU': 0.684,  'F1': 0.806,  'Recall': 0.945,  'Precision': 0.706},
    {'Model': 'ResNet34-UNet 192×192 (best)',  'Signal': 'spectral+texture',  'IoU': 0.8397, 'F1': 0.9055, 'Recall': 0.9585, 'Precision': 0.8579},
]

# ── Add SAM + structural results from LOSO evaluation ─────────────────────────
if not loso_df.empty:
    sam_row = {
        'Model'    : 'SAM + Structural Features (XGBoost)',
        'Signal'   : 'morphological',
        'IoU'      : round(loso_df['IoU'].mean(),      4),
        'F1'       : round(loso_df['F1'].mean(),       4),
        'Recall'   : round(loso_df['Recall'].mean(),   4),
        'Precision': round(loso_df['Precision'].mean(),4),
    }
else:
    sam_row = {
        'Model': 'SAM + Structural Features (XGBoost)',
        'Signal': 'morphological',
        'IoU': None, 'F1': None, 'Recall': None, 'Precision': None,
    }

comparison_df = pd.DataFrame(BASELINES + [sam_row])
comparison_df = comparison_df.sort_values('IoU', ascending=False).reset_index(drop=True)

# ── Highlight SAM row ─────────────────────────────────────────────────────────
def highlight_sam(row):
    return ['background-color: #E8F5E9; font-weight: bold'
            if row['Signal'] == 'morphological' else '' for _ in row]

display(comparison_df.style
    .apply(highlight_sam, axis=1)
    .format({'IoU': '{:.4f}', 'F1': '{:.4f}', 'Recall': '{:.4f}', 'Precision': '{:.4f}'})
    .set_caption('ShrubMap — Model Comparison (IoU = primary metric)')
)

comparison_df.to_csv(OUT_DIR / 'model_comparison.csv', index=False)
print('Saved: model_comparison.csv')

In [ ]:
# ── CELL 15 : Comparison Bar Chart ────────────────────────────────────────────
# Visualises the full model comparison across IoU, F1, and Recall.
# The SAM + structural bar is highlighted in green to distinguish it from
# spectral-only approaches.

metrics = ['IoU', 'F1', 'Recall', 'Precision']
n_models = len(comparison_df)
x = np.arange(n_models)
bar_w = 0.20

fig, ax = plt.subplots(figsize=(16, 6))

colors_map = {
    'spectral'         : '#5B9BD5',
    'spectral+texture' : '#4472C4',
    'morphological'    : '#2E7D32',
}

for k, metric in enumerate(metrics):
    vals  = comparison_df[metric].fillna(0).values
    cols  = [colors_map[s] for s in comparison_df['Signal']]
    bars  = ax.bar(x + k * bar_w, vals, bar_w, label=metric, color=cols, alpha=0.85, edgecolor='white')

ax.set_xticks(x + bar_w * 1.5)
ax.set_xticklabels(comparison_df['Model'], rotation=25, ha='right', fontsize=8)
ax.set_ylabel('Score', fontsize=11)
ax.set_ylim(0, 1.05)
ax.set_title('ShrubMap — Full Model Comparison\nIoU · F1 · Recall · Precision', fontsize=13)
ax.grid(axis='y', alpha=0.3)

# Legend: signal types
legend_patches = [
    mpatches.Patch(color='#5B9BD5', label='Spectral only'),
    mpatches.Patch(color='#4472C4', label='Spectral + Texture'),
    mpatches.Patch(color='#2E7D32', label='Morphological (this work)'),
]
# Legend: metrics
metric_handles = [mpatches.Patch(color='none', label=m) for m in metrics]
ax.legend(handles=legend_patches + metric_handles, ncol=4, fontsize=8, loc='upper right')

plt.tight_layout()
plt.savefig(OUT_DIR / 'model_comparison_plot.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: model_comparison_plot.png')

In [ ]:
# ── CELL 16 : Summary & Next Steps ───────────────────────────────────────────
# Prints a concise summary of findings and actionable next steps.

sam_iou = loso_df['IoU'].mean() if not loso_df.empty else float('nan')
xgb_baseline_iou = 0.672   # From notebook 03
unet_best_iou    = 0.8397  # From notebook 04

print('=' * 68)
print(' NOTEBOOK 05 — SAM SEGMENTATION + STRUCTURAL FEATURES — SUMMARY')
print('=' * 68)

print(f"""
Approach
--------
  Segment Anything Model (zero-shot) generates candidate object masks from
  NAIP RGB tiles.  Each mask is described by 18 structural/morphological
  features (area, compactness, elongation, convexity, fractal dimension,
  Hu moments, SAM confidence scores).  XGBoost classifies each mask as
  shrub or background.  Predicted-shrub masks are unioned to reconstruct
  a pixel-level segmentation map.

Key Results  (leave-one-site-out, pixel level)
--------------
  SAM + Structural (this notebook) :  IoU = {sam_iou:.4f}
  XGBoost on 12 spectral channels  :  IoU = {xgb_baseline_iou:.4f}  (notebook 03)
  ResNet34-UNet 192×192 (best)     :  IoU = {unet_best_iou:.4f}  (notebook 04)

Interpretation
--------------
  SAM operates without any spectral or domain-specific training, relying
  entirely on RGB contrast and shape continuity.  Its performance relative
  to spectral classifiers quantifies how much discriminative information
  exists in shrub morphology alone.

  SHAP analysis reveals the structural features that drive shrub
  identification — compactness and solidity tend to dominate, confirming
  that shrubs are compact, roughly convex objects at NAIP resolution.

Recommended Next Steps
----------------------
  1. Fusion: concatenate SAM structural features with spectral patch
     features from notebook 01 → train a combined XGBoost / deep model.
     Expected gain: structural features provide complementary signal.

  2. SAM 2 (Hiera backbone): try the updated SAM 2 checkpoint for higher
     quality boundaries on fine-resolution imagery.

  3. Point prompts from LiDAR centroids: replace the automatic grid with
     TLS-derived shrub centroid prompts to improve SAM recall on small
     or dense shrub clusters.

  4. SAM mask features as an extra channel in the UNet: append SAM
     boundary distance maps as a 13th input channel to the ResNet34-UNet.

  5. Hyperparameter tuning: tune SAM_AMG_KWARGS (points_per_side,
     pred_iou_thresh) on a held-out Sedgwick validation tile.
""")
print('=' * 68)